# Concatenate Multi-Recording EDF Sessions

Find session folders that contain multiple EDF files with `_recording` in the filename, concatenate those recordings, and write the concatenated EDF file beside the source recordings in the configured raw-data root.


## 1. Imports

Install missing packages in the notebook environment if needed:

```bash
pip install mne pandas edfio
```


In [ ]:
from pathlib import Path
import re

import pandas as pd

try:
    import mne
except ImportError as exc:
    raise ImportError("Install MNE first: pip install mne") from exc

from hypnose_eeg.io.repository_paths import get_rawdata_root
from hypnose_eeg.preprocessing.concatenate_recordings import EdfSessionConcatenator

mne.set_log_level("INFO")


## 2. Source Location

`SOURCE_DIR` comes from the active machine-specific data-location profile. Concatenated files are written into the same session/ephys folder as the source EDF recordings.


In [ ]:
SOURCE_DIR = get_rawdata_root()

EDF_PATTERN = "**/*.edf"
OVERWRITE = False

if not SOURCE_DIR.exists():
    raise FileNotFoundError(f"Source directory not found: {SOURCE_DIR.resolve()}")

concatenator = EdfSessionConcatenator(
    source_dir=SOURCE_DIR,
    edf_pattern=EDF_PATTERN,
    overwrite=OVERWRITE,
)

SOURCE_DIR.resolve()


## 3. Find Multi-Recording Sessions

A session is treated as multi-recording when the same folder contains more than one `.edf` file whose filename includes `_recording`.


In [ ]:
multi_recording_sessions = concatenator.find_multi_recording_sessions()

session_rows = []
for session_dir, files in multi_recording_sessions.items():
    session_rows.append({
        "session_path": str(session_dir.relative_to(SOURCE_DIR)),
        "n_recordings": len(files),
        "recordings": [file.name for file in files],
    })

multi_session_df = pd.DataFrame(session_rows).sort_values("session_path") if session_rows else pd.DataFrame(
    columns=["session_path", "n_recordings", "recordings"]
)
multi_session_df


## 4. Concatenate and Export

The `EdfSessionConcatenator` class writes one EDF output into the source session folder and inserts `CONCAT_BOUNDARY ...` annotations at every join point.


In [ ]:
# Output paths for the first detected session, useful for checking the naming convention.
if multi_recording_sessions:
    preview_session_dir, preview_files = next(
        iter(sorted(multi_recording_sessions.items(), key=lambda item: str(item[0])))
    )
    edf_output_path = concatenator.output_path(preview_session_dir, preview_files)
    pd.Series({
        "session_path": str(preview_session_dir.relative_to(SOURCE_DIR)),
        "edf_output_path": str(edf_output_path),
    })
else:
    print("No multi-recording EDF sessions found.")


## 5. Run

Set `DRY_RUN = False` to write concatenated EDF files. Leave it as `True` to preview output paths first.


In [ ]:
DRY_RUN = True

if DRY_RUN:
    results = concatenator.preview()
else:
    results = concatenator.concatenate_all()

results_df = pd.DataFrame([result.__dict__ for result in results])
results_df


## 6. Create First Multi-Recording Session

This cell writes the first detected multi-recording session beside the source EDFs in EDF format. If the output already exists and `OVERWRITE = False`, it reports `skipped_exists`.


In [ ]:
first_session_result = concatenator.concatenate_first()
first_session_result_df = pd.DataFrame([first_session_result.__dict__])
first_session_result_df


## 7. Save Manifest

The manifest is written only when `results_df` has rows.


In [ ]:
if results_df.empty:
    print("No multi-recording EDF sessions found.")
else:
    manifest_path = concatenator.write_manifest(results, SOURCE_DIR / "edf_concatenation_manifest.csv")
    manifest_path.resolve()
